# AlphaFold2 Structure Prediction: ω-Conotoxin

This notebook runs ColabFold (AlphaFold2-PTM) on ω-conotoxin sequences:
- **SA strong-seeded** (50 SA-generated, conditioned on Cav2.2 binders)
- **SA full-seeded** (50 SA-generated, full-family seeding)

**Stored (natural) sequences are NOT included.** They are unaffected by the pipeline fix
that made this rerun necessary (see below) and were verified byte-identical to the
sequences already scored in `data/af2_results/omega_conotoxin/conotoxin_af2_validation_raw_corrected.csv`
after accounting for ColabFold's header sanitization (`|` and `/` -> `_`). Reuse those
50 rows rather than resubmitting them.

**Why this rerun is needed:** the `generated_strong_seeded.fasta`/`generated_full_seeded.fasta`
sequences changed for two independent, verified reasons since the existing AF2 results were
computed: (1) a fix to how the designated ("strong binder") subset was aligned before
generation, and (2) a fix to which 50 of the 1,550 generated sequences per source are
selected for structural validation (one representative state per sampling chain, matching
what `run_conotoxin_structure_validation.jl` now submits to ESMFold, instead of the first 50
sequences which sampled only 1-2 of 50 chains). Both fixes were independently verified against
the live repository state before this notebook was updated.

Reference structure: 1OMG chain A (ω-conotoxin MVIIA, NMR)

**Requirements:** Run on Google Colab with a **T4 GPU** runtime.

---

## 1. Install ColabFold

In [ ]:
%%bash
set -euo pipefail
# Remove TF to avoid JAX/TF kernel conflicts in Colab
pip uninstall -q -y tensorflow tensorflow-gpu tf-keras 2>/dev/null || true

# Install ColabFold (takes ~3-5 minutes)
if ! command -v colabfold_batch &> /dev/null; then
    pip install -q "colabfold[alphafold] @ git+https://github.com/sokrypton/ColabFold"
    pip install -q "jax[cuda12]"
fi
colabfold_batch --help | head -3

# Record the exact installed version for reproducibility. The original AF2 results in
# this repo do not have a recorded ColabFold version (unpinned git install), so exact
# reproducibility with that earlier run cannot be guaranteed regardless; recording it now
# at least makes this run's provenance traceable if it needs to be repeated later. Current
# ColabFold also writes a per-query config.json with the exact commit; those are preserved
# alongside the PDBs in the results zip (see the run and download cells below).
pip show colabfold 2>/dev/null | grep -E "^(Name|Version|Location)"

## 2. Upload Input FASTA Files

Upload the two FASTA files from `code/data/af2_input/omega_conotoxin/`:
- `sa_strong.fasta`
- `sa_full.fasta`

Do NOT upload `stored.fasta` — the stored (natural) group is reused from the existing
results and is not part of this rerun.

In [ ]:
import os
import hashlib
from google.colab import files

# Create working directories
WORK = "/content/conotoxin_af2"
os.makedirs(f"{WORK}/input", exist_ok=True)
os.makedirs(f"{WORK}/output", exist_ok=True)
os.makedirs(f"{WORK}/results", exist_ok=True)

# SHA-256 of the current (post-fix) code/data/af2_input/omega_conotoxin/*.fasta, recomputed
# and cross-checked (Julia selection, Python selection, and this exact file) immediately
# before this notebook was finalized. Checking the hash, not just the filename and sequence
# count, is required: the old pre-fix FASTAs also have these exact filenames and also
# contain exactly 50 sequences each, so a filename/count check alone would silently accept
# stale input and waste this entire run.
EXPECTED_SHA256 = {
    "sa_strong.fasta": "a450f5542e2f6cc5cf31b8d65fa96c4d88992cda5b3110a9f81261aa8ed0e09e",
    "sa_full.fasta": "735118ea277e06b24dfccfcd6389fa1c9d3206a275b419b64d3deec8da494166",
}

print("Upload the 2 FASTA files (sa_strong.fasta, sa_full.fasta) — NOT stored.fasta:")
uploaded = files.upload()
expected = set(EXPECTED_SHA256)
if set(uploaded.keys()) != expected:
    raise ValueError(
        f"Expected exactly {expected}, got {set(uploaded.keys())}. "
        "This rerun intentionally excludes stored.fasta (reused from existing results)."
    )

# Names actually uploaded per category, captured here so the run cell below can verify
# ColabFold's output against exactly what was submitted rather than a re-derived guess.
UPLOADED_NAMES = {}
for fname, content in uploaded.items():
    digest = hashlib.sha256(content).hexdigest()
    if digest != EXPECTED_SHA256[fname]:
        raise ValueError(
            f"{fname}: SHA-256 {digest} does not match the expected "
            f"{EXPECTED_SHA256[fname]}. This is almost certainly a stale or wrong file "
            "(e.g. re-uploaded from an old download) — re-generate it with "
            "`python3 experiments/prepare_af2_input.py` and upload the fresh copy. "
            "Do not proceed with a hash mismatch."
        )
    with open(f"{WORK}/input/{fname}", 'wb') as f:
        f.write(content)
    text = content.decode()
    names = {line[1:].strip() for line in text.splitlines() if line.startswith('>')}
    n_seqs = len(names)
    print(f"  {fname}: {n_seqs} sequences, SHA-256 verified")
    if n_seqs != 50:
        raise ValueError(f"{fname} has {n_seqs} sequences, expected exactly 50.")
    cat = fname.replace('.fasta', '')
    UPLOADED_NAMES[cat] = names

## 3. Download Reference Structure (1OMG)

In [ ]:
import urllib.request

# Download 1OMG (MVIIA NMR structure)
REF_PDB = f"{WORK}/1OMG.pdb"
urllib.request.urlretrieve(
    "https://files.rcsb.org/download/1OMG.pdb",
    REF_PDB
)

# Extract chain A only
REF_PDB_A = f"{WORK}/1OMG_A.pdb"
with open(REF_PDB) as fin, open(REF_PDB_A, 'w') as fout:
    for line in fin:
        if (line.startswith('ATOM') or line.startswith('TER')) and len(line) > 21 and line[21] == 'A':
            fout.write(line)
        elif line.startswith('END'):
            fout.write(line)

print(f"Reference: {REF_PDB_A}")

## 4. Install TM-align

In [ ]:
%%bash
if ! command -v TMalign &> /dev/null; then
    wget -q https://zhanggroup.org/TM-align/TMalign.cpp -O /tmp/TMalign.cpp
    g++ -O3 -o /usr/local/bin/TMalign /tmp/TMalign.cpp
fi
TMalign -h 2>&1 | head -3

## 5. Run ColabFold AF2-PTM

This runs AlphaFold2-PTM on 100 sequences (50 sa_strong + 50 sa_full; stored is excluded,
~15-20 min on T4).

In [ ]:
import subprocess
import time

# "stored" intentionally excluded: unaffected by the pipeline fix, reused from existing
# data/af2_results/omega_conotoxin/conotoxin_af2_validation_raw_corrected.csv.
CATEGORIES = ["sa_strong", "sa_full"]

for cat in CATEGORIES:
    fasta = f"{WORK}/input/{cat}.fasta"
    outdir = f"{WORK}/output/{cat}"
    os.makedirs(outdir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"Running ColabFold on: {cat}")
    print(f"{'='*60}")

    t0 = time.time()
    result = subprocess.run([
        "colabfold_batch",
        fasta,
        outdir,
        "--model-type", "alphafold2_ptm",
        "--num-models", "1",
        "--num-recycle", "3",
        "--model-order", "1",
        "--num-relax", "0",
        "--msa-mode", "mmseqs2_uniref_env",
        # Explicit rather than relying on current defaults, so this run's determinism
        # doesn't silently drift if ColabFold's own defaults change later.
        "--random-seed", "0",
        "--num-seeds", "1",
    ], capture_output=True, text=True)

    elapsed = time.time() - t0
    print(f"  colabfold_batch finished in {elapsed:.0f}s, returncode={result.returncode}")

    # ColabFold can catch and skip individual-query failures internally and still exit 0,
    # so a clean returncode alone does not mean all 50 sequences were folded. Require an
    # exact match against what was actually uploaded before trusting this category's output.
    if result.returncode != 0:
        print(result.stdout[-2000:] if result.stdout else "no stdout")
        print(result.stderr[-2000:] if result.stderr else "no stderr")
        raise RuntimeError(
            f"colabfold_batch returned {result.returncode} for {cat}. "
            "Stopping rather than continuing on a partial/failed run."
        )

    pdb_files = [f for f in os.listdir(outdir) if f.endswith('.pdb') and 'unrelaxed' in f]
    if not pdb_files:
        pdb_files = [f for f in os.listdir(outdir) if f.endswith('.pdb')]
    output_names = set()
    for f in pdb_files:
        n = f.split('_unrelaxed_')[0] if '_unrelaxed_' in f else f.replace('.pdb', '')
        output_names.add(n)

    expected_names = UPLOADED_NAMES[cat]
    if len(pdb_files) != 50 or output_names != expected_names:
        missing = expected_names - output_names
        extra = output_names - expected_names
        raise RuntimeError(
            f"{cat}: expected exactly 50 PDBs matching the uploaded sequence names, got "
            f"{len(pdb_files)} PDBs. Missing: {sorted(missing)}. Unexpected: {sorted(extra)}. "
            "ColabFold likely failed on some sequences without a nonzero exit code — "
            "do not treat this category's output as complete."
        )
    print(f"  Verified: {len(pdb_files)}/50 PDBs present, all names match the upload.")

print("\nAll categories completed with exactly 50/50 verified PDBs.")

## 6. Extract pLDDT and Compute TM-scores

In [ ]:
import re
import csv
import numpy as np

def extract_mean_plddt(pdb_path):
    """Extract mean pLDDT from B-factor column of ATOM records."""
    plddts = []
    with open(pdb_path) as f:
        for line in f:
            if line.startswith('ATOM') and len(line) >= 66:
                try:
                    plddt = float(line[60:66].strip())
                    plddts.append(plddt)
                except ValueError:
                    pass
    if not plddts:
        return None
    m = np.mean(plddts)
    # ColabFold outputs pLDDT in [0,100]; ESMFold in [0,1]
    return m * 100.0 if m < 1.5 else m


def run_tmalign(query_pdb, ref_pdb):
    """Run TM-align and return TM-score normalized by reference length."""
    result = subprocess.run(
        ["TMalign", query_pdb, ref_pdb],
        capture_output=True, text=True
    )
    for line in result.stdout.split('\n'):
        if 'TM-score=' in line and 'Chain_2' in line:
            m = re.search(r'TM-score=\s*([\d.]+)', line)
            if m:
                return float(m.group(1))
    return None


# Collect results
results = []
for cat in CATEGORIES:
    outdir = f"{WORK}/output/{cat}"
    pdbs = sorted([f for f in os.listdir(outdir) if f.endswith('.pdb') and 'unrelaxed' in f])

    # If no 'unrelaxed' in name, try all PDBs
    if not pdbs:
        pdbs = sorted([f for f in os.listdir(outdir) if f.endswith('.pdb')])

    print(f"\n{cat}: {len(pdbs)} PDB files")
    for pdb_file in pdbs:
        pdb_path = os.path.join(outdir, pdb_file)
        plddt = extract_mean_plddt(pdb_path)
        tmscore = run_tmalign(pdb_path, REF_PDB_A)

        # Extract sequence name from filename
        name = pdb_file.replace('_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb', '')
        name = name.replace('.pdb', '')

        if plddt is None or tmscore is None or not np.isfinite(plddt) or not np.isfinite(tmscore):
            raise RuntimeError(
                f"{cat}/{name}: failed to extract a finite pLDDT/TM-score "
                f"(plddt={plddt}, tmscore={tmscore}) from {pdb_path}. A PDB existing does "
                "not guarantee scoring succeeded — do not silently drop this row."
            )

        results.append({
            'name': name,
            'source': cat,
            'plddt': plddt,
            'tmscore': tmscore,
            'predictor': 'AF2'
        })
        print(f"  {name}: pLDDT={plddt:.1f}, TM={tmscore:.4f}")

# Final completeness check: exactly 50 unique, finite rows per category, matching the
# verified upload names (already checked once at the PDB-file level in the run cell above;
# re-checking here catches anything that could slip through between the two stages).
for cat in CATEGORIES:
    subset_names = {r['name'] for r in results if r['source'] == cat}
    if len(subset_names) != 50 or subset_names != UPLOADED_NAMES[cat]:
        raise RuntimeError(f"{cat}: final extracted results do not match the 50 uploaded names.")
if len(results) != 100:
    raise RuntimeError(f"Expected 100 total rows (50+50), got {len(results)}.")
print(f"\nVerified: {len(results)} rows, 50 per category, all finite, all names matched.")

# Save raw results
csv_path = f"{WORK}/results/conotoxin_af2_validation_raw.csv"
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['name', 'source', 'plddt', 'tmscore', 'predictor'])
    writer.writeheader()
    writer.writerows(results)
print(f"Raw results saved to: {csv_path}")

## 7. Summary Statistics

In [ ]:
print(f"{'Source':<15} {'n':>3}  {'pLDDT':>15}  {'TM-score':>15}")
print("-" * 55)
for cat in CATEGORIES:
    subset = [r for r in results if r['source'] == cat]
    plddts = [r['plddt'] for r in subset if r['plddt'] is not None]
    tms = [r['tmscore'] for r in subset if r['tmscore'] is not None]
    print(f"{cat:<15} {len(subset):>3}  "
          f"{np.mean(plddts):>6.1f} ± {np.std(plddts):>4.1f}  "
          f"{np.mean(tms):>6.4f} ± {np.std(tms):.4f}")

# Save summary
summary_path = f"{WORK}/results/conotoxin_af2_validation_summary.csv"
with open(summary_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['source', 'n', 'plddt_mean', 'plddt_std', 'tmscore_mean', 'tmscore_std'])
    for cat in CATEGORIES:
        subset = [r for r in results if r['source'] == cat]
        plddts = [r['plddt'] for r in subset if r['plddt'] is not None]
        tms = [r['tmscore'] for r in subset if r['tmscore'] is not None]
        writer.writerow([cat, len(subset), f"{np.mean(plddts):.1f}", f"{np.std(plddts):.1f}",
                        f"{np.mean(tms):.4f}", f"{np.std(tms):.4f}"])
print(f"\nSummary saved to: {summary_path}")

## 8. Download Results

In [ ]:
# Download CSV results
files.download(f"{WORK}/results/conotoxin_af2_validation_raw.csv")
files.download(f"{WORK}/results/conotoxin_af2_validation_summary.csv")

# Also zip all PDB files for download
import shutil
shutil.make_archive(f"{WORK}/results/conotoxin_af2_pdbs", 'zip', f"{WORK}/output")
files.download(f"{WORK}/results/conotoxin_af2_pdbs.zip")
print("Done! Download the CSV files and PDB zip.")